# Customer Segmentation and Analysis 

Steps to solve the problem :
1. Importing Libraries.
2. Exploration of data.
3. Data Visualization.
4. Clustering using K-Means.
5. Selection of Clusters.
6. Ploting the Cluster Boundry and Clusters.
7. 3D Plot of Clusters.

## Importing Libraries
### Import all the libraries required for exploration, visualization, algorithm.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")


## Data Exploration
### Explore the data (head, shape, description, data types, check for null values)

In [ ]:
df = pd.read_csv("customers.csv")
print(df.shape)
df.head()


## Data Visualization
    Using the imported libraries

In [ ]:
print(df.describe(include="all"))
print("\nmissing values per column:")
print(df.isnull().sum())
print("\ndtypes:")
print(df.dtypes)


### Count Plot of Gender
    Using seaborn count plot

In [ ]:
sns.countplot(data=df, x="Gender")
plt.title("Count of customers by gender")
plt.show()


### Histograms
    Plot histogram of the features (Age, Annual Income and Spending Score)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Age", "Annual Income (k$)", "Spending Score (1-100)"]):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()


### Ploting the Relation between Age , Annual Income and Spending Score
    Using seaborn regplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.regplot(data=df, x="Age", y="Annual Income (k$)", ax=axes[0],
            scatter_kws={"alpha": 0.5})
sns.regplot(data=df, x="Age", y="Spending Score (1-100)", ax=axes[1],
            scatter_kws={"alpha": 0.5}, color="seagreen")
axes[0].set_title("Age vs Annual Income"); axes[1].set_title("Age vs Spending Score")
plt.tight_layout()
plt.show()


### Scatter Plot Age: vs Annual Income w.r.t Gender

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="Age", y="Annual Income (k$)", hue="Gender", s=60)
plt.title("Age vs Annual Income, coloured by gender")
plt.show()


### Scatter Plot: Annual Income vs Spending Score w.r.t Gender

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
                hue="Gender", s=60)
plt.title("Annual Income vs Spending Score, coloured by gender")
plt.show()


### Distribution of values in Age , Annual Income and Spending Score according to Gender
Violinplot and Swarmplot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["Age", "Annual Income (k$)", "Spending Score (1-100)"]):
    sns.violinplot(data=df, x="Gender", y=col, ax=ax, inner=None, color="lightgrey")
    sns.swarmplot(data=df, x="Gender", y=col, ax=ax, size=3, color="black")
    ax.set_title(col)
plt.tight_layout()
plt.show()


## Clustering using K- means

1. Use elbow method to calculate number of cluster

2. Visualize the elbow curve to decide the number of clusters.

3. Create and the model using KMeans for the calculated number of clusters.

4. Apply the Model and visualize cluster using a scater plot

### 1.Segmentation using Age and Spending Score

In [ ]:
X = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].to_numpy()

wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    wcss.append(km.inertia_)

plt.figure(figsize=(7, 4.5))
plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-cluster sum of squares (inertia)")
plt.title("Elbow method (Age, Annual Income, Spending Score)")
plt.show()


### Selecting N Clusters based in Inertia (Squared Distance between Centroids and data points, should be less)

In [ ]:
# The elbow is a judgement call, but the curve bends noticeably around k=5-6.
# We'll go with 6, which is also the number of visually distinct groups you can
# see once we plot two features at a time below.
N_CLUSTERS = 6
kmeans_full = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
df["cluster"] = kmeans_full.fit_predict(X)

print(f"Chosen k = {N_CLUSTERS}, inertia = {kmeans_full.inertia_:.1f}")
df.groupby("cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)


### 3.Segmentation using Age , Annual Income and Spending Score

In [ ]:
# Segmentation using just Age and Spending Score, which is what the
# "Assignment" notebook (Annual Income + Spending Score) asks you to redo
# with a different pair of features.
X_age_ss = df[["Age", "Spending Score (1-100)"]].to_numpy()

kmeans_2d = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_2d = kmeans_2d.fit_predict(X_age_ss)

plt.figure(figsize=(8, 6))
for cluster_id in range(4):
    mask = labels_2d == cluster_id
    plt.scatter(X_age_ss[mask, 0], X_age_ss[mask, 1], s=50, label=f"cluster {cluster_id}")
plt.scatter(kmeans_2d.cluster_centers_[:, 0], kmeans_2d.cluster_centers_[:, 1],
            s=250, c="black", marker="X", label="centroids")
plt.xlabel("Age"); plt.ylabel("Spending Score (1-100)")
plt.title("Customer segments: Age vs Spending Score")
plt.legend()
plt.show()

print("\nCluster sizes:")
print(pd.Series(labels_2d).value_counts().sort_index())
